# Kelly criterion — companion notebook

Companion for [`kelly-criterion.md`](./kelly-criterion.md). Repeated coin-bet simulation: terminal wealth distributions vs the fraction bet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

In [ ]:
# Coin: p_win = 0.55, even-money (b=1)  => f* = 0.10
p, b, T, M = 0.55, 1.0, 500, 5000
f_star = (b*p - (1-p))/b
fractions = [0.5*f_star, f_star, 1.5*f_star, 2.0*f_star, 5.0*f_star]

def simulate(f, T=T, M=M):
    wins = rng.binomial(1, p, size=(M, T)).astype(bool)
    rets = np.where(wins, 1 + f*b, 1 - f)
    return np.cumprod(rets, axis=1)

wealth = {f: simulate(f) for f in fractions}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for f in fractions:
    W = wealth[f]
    med = np.median(W, axis=0); p5 = np.percentile(W, 5, axis=0); p95 = np.percentile(W, 95, axis=0)
    axes[0].semilogy(med, label=f'f={f:.2f}{" (Kelly)" if abs(f-f_star)<1e-9 else ""}')
print('Stats at T={}:'.format(T))
print(f'{"f":>6}  {"median W":>12}  {"mean log W":>12}  {"P(W<1)":>8}  {"frac > Kelly median":>20}')
for f in fractions:
    W = wealth[f][:, -1]
    print(f'{f:6.3f}  {np.median(W):12.3f}  {np.mean(np.log(W)):12.4f}  {np.mean(W<1):8.2%}')
axes[0].set_xlabel('round'); axes[0].set_ylabel('median wealth (log)'); axes[0].legend(); axes[0].set_title('Median wealth trajectory')

# Terminal log-wealth distributions
for f in fractions:
    axes[1].hist(np.log(wealth[f][:, -1]), bins=60, histtype='step', label=f'f={f:.2f}')
axes[1].axvline(0, color='k', lw=0.5)
axes[1].set_xlabel(f'log W_T (T={T})'); axes[1].set_ylabel('count'); axes[1].legend()
axes[1].set_title('Terminal log-wealth distribution')
plt.tight_layout(); plt.show()